In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import DoubleType

In [0]:
%run /Workspace/Data_Ingestion_Atlikon/utilities

In [0]:
bronze_schema, silver_schema, gold_schema

('bronze', 'silver', 'gold')

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

landing_path = f"s3://childcompanydata-myfirstproject/{data_source}/landing/"
processed_path = f"s3://childcompanydata-myfirstproject/{data_source}/processed/"

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
landing_path, processed_path, bronze_table, silver_table, gold_table

('s3://childcompanydata-myfirstproject/orders/landing/',
 's3://childcompanydata-myfirstproject/orders/processed/',
 'fmcg.bronze.orders',
 'fmcg.silver.orders',
 'fmcg.gold.sb_fact_orders')

# Bronze Layer Setup

In [0]:
try:
    # Block 1: Read
    display(dbutils.fs.ls(landing_path))
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", True)
          .load(landing_path)
          .withColumn("read_timestamp", F.current_timestamp())
          .select("*", "_metadata.file_name", "_metadata.file_size"))

    df = df.withColumn("order_qty", F.col("order_qty").cast(DoubleType()))
    display(df.limit(5))

    # Block 2: Bronze
    df.write.mode("append")\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .saveAsTable(bronze_table)

    # Block 3: Staging
    df.write.mode("overwrite")\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

    # Block 4: ONLY move files on success
    # for file in dbutils.fs.ls(landing_path):
    #     dbutils.fs.mv(file.path, f"{processed_path}/{file.name}", True)
        
    print("Pipeline succeeded ✅")

except Exception as e:
    print(f"Pipeline failed: {e}")
    print("Files NOT moved - check landing_path")


path,name,size,modificationTime
s3://childcompanydata-myfirstproject/orders/landing/orders_2025_12_31.csv,orders_2025_12_31.csv,8008,1767281433000


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891203,318.0,2026-01-01T15:31:08.168Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891502,233.0,2026-01-01T15:31:08.168Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891201,324.0,2026-01-01T15:31:08.168Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1INVALID,25891303,34.0,2026-01-01T15:31:08.168Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1INVALID,25891303,34.0,2026-01-01T15:31:08.168Z,orders_2025_12_31.csv,8008


Pipeline succeeded ✅


# Silver Layer Setup

In [0]:
df_orders = spark.sql(f"select * from {catalog}.{bronze_schema}.staging_{data_source}")
display(df_orders)

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891203,318.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891502,233.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891201,324.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1INVALID,25891303,34.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1INVALID,25891303,34.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,"Tuesday, December 30, 2025",1789720,25891502,233.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,30-12-2025,1INVALID,25891102,418.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,30-12-2025,1789321,99999999,231.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,"Tuesday, December 30, 2025",1789321,25891201,484.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,"Tuesday, December 30, 2025",1789321,25891603,88.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008


## Transformations
- order_qty is not NULL
- Remove weekday from order_placement_day
- Type cast date to standard format
- Replace invalid customer_id with 999999
- Drop duplicates based on required columns
- Convert product_id to string


In [0]:
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.when(F.col("order_placement_date").rlike(r"^[A-Za-z]+,\s*"), F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", ""))
    .otherwise(F.col("order_placement_date"))
)

df_orders = df_orders.withColumn(
   "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
    .otherwise("999999")
    .cast("string")
)

df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))


In [0]:
display(df_orders)

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
ABFDEC831720502,2025-12-30,1789720,25891203,318.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,2025-12-30,1789720,25891502,233.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,2025-12-30,1789720,25891201,324.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831720502,2025-12-30,999999,25891303,34.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,2025-12-30,999999,25891102,418.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,2025-12-30,1789321,99999999,231.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,2025-12-30,1789321,25891201,484.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831321603,2025-12-30,1789321,25891603,88.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831702601,2025-12-30,1789702,25891103,344.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008
ABFDEC831702601,2025-12-30,1789702,25891601,139.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008


In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-12-30|2025-12-30|
+----------+----------+



In [0]:
df_products = spark.sql(f"select * from {catalog}.{silver_schema}.products")
df_orders = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])
display(df_orders.limit(5))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,product_code
ABFDEC831522303,2025-12-30,1789522,25891303,61.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
ABFDEC831420303,2025-12-30,1789420,25891303,19.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
ABFDEC831903302,2025-12-30,1789903,25891302,55.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
ABFDEC831520501,2025-12-30,1789520,25891301,85.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
ABFDEC831122203,2025-12-30,1789122,25891203,388.0,2026-01-01T15:31:12.581Z,orders_2025_12_31.csv,8008,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2


In [0]:
if not spark.catalog.tableExists(silver_table):
    spark.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta= DeltaTable.forName(spark, silver_table)
    silver_delta.alias("t").merge(
        df_orders.alias("s"), "s.order_id = t.order_id and s.order_placement_date = t.order_placement_date and s.customer_id = t.customer_id and s.order_qty = t.order_qty"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_orders.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

# Gold Layer Setup

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {catalog}.{silver_schema}.staging_{data_source};")

df_gold.show(2)

+---------------+----------+-------------+--------------------+----------+-------------+
|       order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+---------------+----------+-------------+--------------------+----------+-------------+
|ABFDEC831522303|2025-12-30|      1789522|c68834ceaff15846b...|  25891303|         61.0|
|ABFDEC831420303|2025-12-30|      1789420|c68834ceaff15846b...|  25891303|         19.0|
+---------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
df_gold.count()

93

In [0]:
print(spark.sql(f"select count(*) from {gold_table};"))

if not spark.catalog.tableExists(gold_table):
    df_gold.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").saveAsTable(gold_table)
else:
    gold_delta= DeltaTable.forName(spark, gold_table)
    gold_delta.alias("t").merge(
        df_gold.alias("s"), "s.order_id = t.order_id and s.date = t.date and s.customer_code = t.customer_code and s.product_code = t.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    
print(spark.sql(f"select count(*) from {gold_table};"))

DataFrame[count(*): bigint]
DataFrame[count(*): bigint]


In [0]:
df_gold = spark.sql(f"SELECT * FROM {gold_table};")

df_gold.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).show()
df_gold.count()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-12-30|
+----------+----------+



48640

In [0]:
# Parent co. data is at month level.
# Whereas child co. data is at daily level. We agg them to month level.
# And merge with parent co. data

df_gold_child = spark.sql(f"select date, product_code, customer_code, sold_quantity from {gold_table}")

df_gold_child_monthly = df_gold_child.withColumn(
    "date", F.trunc("date", "MM")
).groupBy(
    "date", "product_code", "customer_code"
).agg(
    F.sum("sold_quantity").alias("sold_quantity")
)

df_gold_child_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-09-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789301       |5104.0       |
|2025-09-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789301       |4640.0       |
|2025-09-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789401       |585.0        |
|2025-10-01|fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49|789903       |3471.0       |
|2025-11-01|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789522       |1569.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_gold_child_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_parent_gold = spark.sql(f"select * from {catalog}.{gold_schema}.fact_orders")
df_parent_gold.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-01-01|2025-12-01|
+----------+----------+



In [0]:
%sql

drop table if exists fmcg.bronze.staging_orders;
drop table if exists fmcg.silver.staging_orders;

In [0]:
df_parent_gold = spark.sql(f"select * from {catalog}.{gold_schema}.fact_orders")
df_parent_gold.printSchema()

display(
    df_parent_gold
    .groupby("date")
    .agg(F.sum(F.col("sold_quantity").cast("double")).alias("total_records"))
    .orderBy("date")
)

root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: long (nullable = true)



date,total_records
2024-01-01,461181.0
2024-02-01,477125.0
2024-03-01,61202.0
2024-04-01,225753.0
2024-05-01,236201.0
2024-06-01,482055.0
2024-07-01,446871.0
2024-08-01,434278.0
2024-09-01,1013218.0
2024-10-01,1374279.0


In [0]:
from pyspark.sql import functions as F

df_parent_gold = spark.sql(f"select * from {catalog}.{gold_schema}.sb_fact_orders")
display(
    df_parent_gold.groupby("date")
    .agg(F.count("*").alias("total_records"))
    .orderBy("date")
)

date,total_records
2025-07-01,286
2025-07-02,264
2025-07-03,254
2025-07-04,271
2025-07-05,273
2025-07-06,270
2025-07-07,264
2025-07-08,268
2025-07-09,253
2025-07-10,260


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df_gold = spark.sql(f"select * from {gold_table}")

In [0]:
display(df_gold.head(5))

order_id,date,customer_code,product_code,product_id,sold_quantity
FDEC825521501,2025-12-24,789521,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,25891401,225.0
FDEC85720203,2025-12-03,789720,2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,25891201,202.0
FDEC822520302,2025-12-19,789520,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,25891302,50.0
FDEC831122203,2025-12-30,789122,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,25891203,393.0
FDEC811520601,2025-12-10,789520,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,25891103,444.0


In [0]:
df_gold.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-12-31|
+----------+----------+



In [0]:
df_gold_final = spark.sql(f"select * from {catalog}.{gold_schema}.fact_orders")

In [0]:
display(df_gold_final.groupBy("product_code").agg(F.min("date").alias("min_date"),(F.max("date").alias("max_date"))))

product_code,min_date,max_date
ARCHAFF0E4,2024-01-01,2025-12-01
BADM9DDCED,2024-01-01,2024-08-01
ROCKD798F6,2024-09-01,2025-12-01
CRICA76614,2024-09-01,2025-12-01
CRICBBFD56,2024-01-01,2025-12-01
SQUA22F29D,2024-01-01,2025-12-01
YOGA1B22A2,2024-01-01,2025-12-01
HIKCAC6DD,2024-01-01,2025-12-01
CRIC2841E6,2024-01-01,2025-12-01
YOGA80ED66,2024-09-01,2025-12-01
